<a href="https://colab.research.google.com/github/GUNAPILLCO/neural_profit/blob/main/stage_04_time_aware_data_splitting/stage_04_time_aware_data_splitting.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **stage_04_time_aware_data_splitting**

## Introducción y Resumen

Esta notebook tiene como objetivo dividir el dataset MNQ en conjuntos de entrenamiento, validación y prueba, asegurando una partición aleatoria, reproducible y estructuralmente consistente. Con ello se dejan listos los datos para el entrenamiento y evaluación de los modelos predictivos.

0. Configuración del Entorno

    Se conecta Google Drive y se clona el repositorio de trabajo. Se instalan e importan librerías necesarias como pandas, numpy, matplotlib y seaborn. Se cargan los datasets procesados previamente (mnq_technical_indicators y mnq_alpha_factors) y se muestra un resumen de la información del dataset MNQ.

1. Carga de datos

    Se importa el dataset procesado con features técnicos y alpha factors. Se revisa su estructura (filas, columnas, tipos de datos) y también de importa el listado de features seleccionados para cada ventana de tiempo.

2. Análisis del dataset `mnq_model`

    Se revisa la estructura del dataset (filas, columnas, tipos de datos), se busca los valores NaNs y se verifica la distribución temporal de los registros.

3. Definición de parámetros de división

    En este punto se define la estrategia de partición del dataset: se toma un 70% de los días para entrenamiento, y el 30% restante se divide en partes iguales para validación y prueba. De esta manera, el modelo cuenta con suficientes datos para aprender, mientras que se reservan bloques temporales separados para ajustar parámetros y evaluar el rendimiento final sin fugas de información.

4. Selección aleatoria de días.

    Este punto busca garantizar que la partición de los datos sea representativa y no esté sesgada por la secuencia temporal. Al asignar los días de forma aleatoria —aunque de manera reproducible— se evita que los conjuntos queden condicionados por períodos específicos del mercado (por ejemplo, tendencias prolongadas o alta volatilidad en ciertos meses). Así, cada subconjunto refleja mejor la diversidad del dataset y se obtiene una evaluación más robusta del modelo.

5. Generación de datasets `mnq_train`, `mnq_test` y `mnq_valid`

    En este punto se crean los datasets mnq_train, mnq_valid y mnq_test, manteniendo homogeneidad en estructura (301 registros por día, de 09:30 a 14:30) y sin solapamiento entre conjuntos. Esto asegura consistencia en el entrenamiento, validación y prueba del modelo.

## 0. Configuración del Entorno


### 0.1. Clonado de repositorio / Acceso a Drive

In [1]:
from google.colab import drive
drive.mount('/content/drive')
drive_path = "/content/drive/MyDrive/neural_profit"

Mounted at /content/drive


### 0.2. Instalación de librerías


In [2]:
#!{sys.executable} -m pip install -q ta
#print("Librería instalada: technical-analysis")

### 0.3. Importación de librerías


In [3]:
import sys
import re
#Instalación de librería pandas_market_calendars
#!{sys.executable} -m pip install -q pandas_market_calendars
#print("Librería instalada: pandas_market_calendars")


from functools import reduce
# Utilidades generales
from datetime import datetime, timedelta
import os
import glob
import requests
import warnings
warnings.filterwarnings('ignore')

# Manejo y procesamiento de datos
#import ta
import pandas as pd
import numpy as np
from tabulate import tabulate
import matplotlib.pyplot as plt
# Calendario de mercados
#import pandas_market_calendars as mcal

#from ta.momentum import StochasticOscillator, ROCIndicator
#from ta.volatility import BollingerBands, AverageTrueRange

from scipy.stats import spearmanr

import os
import json
import logging
from pathlib import Path
from typing import Dict, Any, List, Tuple

import numpy as np
import pandas as pd

#from ta.momentum import ROCIndicator



In [4]:
# ============================================================
# Paths / IO (via env o defaults)
# ============================================================

DRIVE_DIR =   Path(os.environ.get("DRIVE_DIR", "/content/drive/MyDrive/neural_profit/"))

IN_PARQUET = DRIVE_DIR / Path(os.environ.get("IN_PARQUET", "data/04_features/mnq_t2.parquet"))
IN_SUMMARY = DRIVE_DIR /  Path(os.environ.get("IN_PARQUET", "data/04_features/mnq_t2_summary.json"))

##############

OUT_SPLITS_SUMMARY = DRIVE_DIR / Path(os.environ.get("OUT_SUMMARY", "data/05_splits/splits_summary.json"))

OUT_PARQUET_T2_TRAIN = DRIVE_DIR / Path(os.environ.get("OUT_PARQUET_T2_TRAIN", "data/05_splits/mnq_t2_train.parquet"))
OUT_PARQUET_T2_VALID =  DRIVE_DIR / Path(os.environ.get("OUT_PARQUET_T2_VALID", "data/05_splits/mnq_t2_valid.parquet"))
OUT_PARQUET_T2_TEST =  DRIVE_DIR / Path(os.environ.get("OUT_PARQUET_T2_TEST", "data/05_splits/mnq_t2_test.parquet"))

## **1. Carga de datos**

### 1.1. Carga de dataset `mnq_delta_*.parquet`




In [5]:
def _ensure_parent_dir(path: Path) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)

def load_mnq_parquet(path: Path):
    os.path.exists(path)
    print("Archivo encontrado en disco. Cargando dataset local...")
    mnq_parquet = pd.read_parquet(path)
    return mnq_parquet

### 1.2. Información de dataset `mnq_t2`



In [6]:
from __future__ import annotations

from dataclasses import dataclass
from typing import Any, Dict, Optional, Tuple

import pandas as pd


def mnq_dataset_info(
    df: pd.DataFrame,
    *,
    name: str = "mnq_raw",
    tz_assume_if_naive: Optional[str] = None,  # ej: "UTC" o "America/New_York"
    day_def: str = "calendar",  # "calendar" (fecha calendario) o "trading" (días con datos)
) -> Dict[str, Any]:
    """
    Resume un dataset OHLCV con DatetimeIndex (ideal para mnq_raw).

    - Si el índice es tz-naive:
        - Si tz_assume_if_naive != None, lo localiza a esa tz.
        - Si no, reporta "tz-naive" (no se puede afirmar horario UTC).
    - Devuelve dict con métricas principales (y lo imprime bonito si se desea).
    """
    if not isinstance(df.index, pd.DatetimeIndex):
        raise TypeError(f"{name}: se requiere DatetimeIndex, recibido: {type(df.index)}")

    idx = df.index

    # --- timezone / UTC info ---
    tzinfo = idx.tz
    if tzinfo is None:
        tz_status = "tz-naive (sin zona horaria)"
        if tz_assume_if_naive:
            idx = idx.tz_localize(tz_assume_if_naive)
            tzinfo = idx.tz
            tz_status = f"localizado como {tzinfo}"
    else:
        tz_status = f"{tzinfo}"

    # --- rango temporal ---
    ts_min = idx.min()
    ts_max = idx.max()

    first_day = ts_min.date()
    last_day = ts_max.date()

    # --- días ---
    if day_def == "calendar":
        total_days = (pd.Timestamp(last_day) - pd.Timestamp(first_day)).days + 1
    elif day_def == "trading":
        total_days = idx.normalize().nunique()
    else:
        raise ValueError("day_def debe ser 'calendar' o 'trading'")

    # --- columnas ---
    columns = list(df.columns)

    # --- checks útiles ---
    n_rows = len(df)
    n_cols = df.shape[1]
    n_missing = int(df.isna().sum().sum())
    missing_by_col = df.isna().sum().to_dict()
    dup_index = int(idx.duplicated().sum())
    is_monotonic = bool(idx.is_monotonic_increasing)

    # Frecuencia estimada (puede fallar si hay huecos grandes)
    freq = pd.infer_freq(idx[: min(50000, len(idx))])  # muestra grande pero acotada

    # Cobertura por día (min/max de hora del día, en tz del índice)
    # (Útil para ver si es 24/7 o horario de sesión)
    tod = pd.Series(idx.time)
    # Convertimos time a minutos del día para resumen robusto
    tod_minutes = pd.Series([t.hour * 60 + t.minute for t in tod])
    typical_minute_min = int(tod_minutes.min())
    typical_minute_max = int(tod_minutes.max())

    # Rango promedio de filas por día (sólo días con datos)
    rows_per_day = df.groupby(idx.normalize()).size()
    rows_per_day_stats = {
        "days_with_data": int(rows_per_day.shape[0]),
        "rows_per_day_min": int(rows_per_day.min()),
        "rows_per_day_p50": float(rows_per_day.median()),
        "rows_per_day_max": int(rows_per_day.max()),
    }

    # Si hay tz, también mostramos rango en UTC
    if tzinfo is not None:
        ts_min_utc = ts_min.tz_convert("UTC")
        ts_max_utc = ts_max.tz_convert("UTC")
        utc_range = (str(ts_min_utc), str(ts_max_utc))
        utc_note = "El índice está tz-aware; el horario UTC es inequívoco."
    else:
        utc_range = None
        utc_note = "El índice es tz-naive; no se puede asegurar si está en UTC sin suposiciones."

    info: Dict[str, Any] = {
        "name": name,
        "shape": (n_rows, n_cols),
        "columns": columns,
        "index_type": type(df.index).__name__,
        "index_tz": tz_status,
        "utc_note": utc_note,
        "datetime_min": str(ts_min),
        "datetime_max": str(ts_max),
        "first_day": str(first_day),
        "last_day": str(last_day),
        "total_days": int(total_days),
        "day_definition": day_def,
        "utc_range_if_applicable": utc_range,
        "is_index_monotonic_increasing": is_monotonic,
        "duplicated_timestamps_in_index": dup_index,
        "inferred_freq_sample": freq,
        "missing_total_cells": n_missing,
        "missing_by_col": missing_by_col,
        "rows_per_day_stats": rows_per_day_stats,
        "time_of_day_minutes_range": {
            "min_minute_of_day": typical_minute_min,
            "max_minute_of_day": typical_minute_max,
        },
    }
    return info


def print_mnq_dataset_info(info: Dict[str, Any]) -> None:
    """Imprime el dict de mnq_dataset_info de forma ordenada."""
    print(f"Dataset: {info['name']}")
    print(f"Shape: {info['shape']}")
    print(f"Columns: {info['columns']}")
    print(f"Index: {info['index_type']} | TZ: {info['index_tz']}")
    print(f"Datetime min/max: {info['datetime_min']}  ->  {info['datetime_max']}")
    print(f"First/Last day: {info['first_day']}  ->  {info['last_day']}")
    print(f"Total days ({info['day_definition']}): {info['total_days']}")
    #print(f"Inferred freq (sample): {info['inferred_freq_sample']}")
    #print(f"Index monotonic increasing: {info['is_index_monotonic_increasing']}")
    #print(f"Duplicated timestamps in index: {info['duplicated_timestamps_in_index']}")
    #print(f"Missing total cells: {info['missing_total_cells']}")
    #print(f"Missing by col: {info['missing_by_col']}")
    #print(f"Rows/day stats: {info['rows_per_day_stats']}")
    print(f"Time-of-day range (minutes): {info['time_of_day_minutes_range']}")
    print(f"UTC note: {info['utc_note']}")
    if info["utc_range_if_applicable"] is not None:
        print(f"UTC range: {info['utc_range_if_applicable'][0]}  ->  {info['utc_range_if_applicable'][1]}")



## **1.3. Carga de mnq e información**




In [7]:
mnq_t2 = load_mnq_parquet(IN_PARQUET)
info_mnq_t2 = mnq_dataset_info(mnq_t2, name="mnq_t2", tz_assume_if_naive=None, day_def="trading")
print_mnq_dataset_info(info_mnq_t2)

Archivo encontrado en disco. Cargando dataset local...
Dataset: mnq_t2
Shape: (88920, 12)
Columns: ['date', 'minute_of_day', 'ema_60', 'roc_60', 'roc_30', 'stoch_k_30', 'mom_5', 'atr_norm_10', 'macd', 't2_p40_h30', 't2_p40_h60', 't2_p50_h30']
Index: DatetimeIndex | TZ: America/New_York
Datetime min/max: 2020-01-02 09:30:00-05:00  ->  2026-04-17 10:29:00-04:00
First/Last day: 2020-01-02  ->  2026-04-17
Total days (trading): 1482
Time-of-day range (minutes): {'min_minute_of_day': 570, 'max_minute_of_day': 629}
UTC note: El índice está tz-aware; el horario UTC es inequívoco.
UTC range: 2020-01-02 14:30:00+00:00  ->  2026-04-17 14:29:00+00:00


In [8]:
import json
import os
from pathlib import Path

# Cargar JSON
with open(IN_SUMMARY, "r") as f:
    summary = json.load(f)

# Extraer columnas
features_col = summary["column_groups"]["features"]
targets_col  = summary["column_groups"]["targets"]
aux_col      = summary["column_groups"]["auxiliary"]

# Check rápido
print("Features:", features_col)
print("Targets:", targets_col)
print("Aux:", aux_col)

Features: ['ema_60', 'roc_60', 'roc_30', 'stoch_k_30', 'mom_5', 'atr_norm_10', 'macd']
Targets: ['t2_p40_h30', 't2_p40_h60', 't2_p50_h30']
Aux: ['date', 'minute_of_day', 'regime_id']


# **2. Análisis de datasets**

## **2.0. Funciones**

### Función para contar NaN en dataset

In [9]:
def nan_count(df: pd.DataFrame, mnq: str) -> None:
    """
    Verifica la existencia de NaN por día y por columna.
    - Si existen NaN, muestra únicamente las columnas afectadas.
    - Si no existen, muestra un mensaje indicando que no hay NaNs.
    """

    # Conteo de NaN por día y columna
    daily_nan_counts = (
        df.groupby("date")
        .apply(lambda x: x.isna().sum())
    )

    # Identificar columnas con al menos un NaN en cualquier día
    cols_with_nan = daily_nan_counts.columns[
        (daily_nan_counts > 0).any(axis=0)
    ].tolist()

    if cols_with_nan:
        print("Columnas con valores NaN:")
        for col in cols_with_nan:
            total_nans = df[col].isna().sum()
            print(f" - {col}: {total_nans} NaNs")
    else:
        print(f"{mnq} sin valores NaNs")

### Función para detectar saltos temporales (gaps)

In [10]:
import pandas as pd

def detectar_gaps(df: pd.DataFrame, mnq: str, gap_minutes: int = 1):
    """
    Verifica si existen saltos mayores al intervalo esperado (por defecto 1 minuto)
    entre registros consecutivos dentro de cada día, en un DataFrame con índice tipo DatetimeIndex.

    Omite el primer registro de cada día.

    Parámetros:
    - df: DataFrame con índice datetime.
    - mnq_delta_h: nombre/identificador del dataset (para mensajes).
    - gap_minutes: tamaño esperado del intervalo en minutos (por defecto 1).

    Retorna:
    - Lista de índices donde se detectaron diferencias mayores al intervalo esperado.
      (por día se guarda un Index con los timestamps irregulares)
    """
    df = df.copy()
    df["time_diff"] = df.index.to_series().diff()

    base_time_diff = pd.Timedelta(minutes=gap_minutes)
    problem_indices = []

    for date, group in df.groupby(df.index.date):
        time_diff = group["time_diff"].iloc[1:]  # omite el primer registro del día
        incorrect_indices = time_diff[time_diff != base_time_diff].index
        if len(incorrect_indices) > 0:
            problem_indices.append(incorrect_indices)

    if problem_indices:
        print(f"{mnq} con gaps")
        print(f"Se encontraron problemas en {sum(len(x) for x in problem_indices)} registros con diferencias irregulares.\n")

        # Conteo por fecha
        conteos = df.groupby(df.index.date).size()

        for day_idx in problem_indices:
            idx = day_idx[0]  # primer timestamp irregular del día
            diff = df.loc[idx, "time_diff"]
            date = idx.date()
            count = conteos[date]
            print(f"\t{idx} -> Diferencia: {diff} | # Registros: {count}")
    else:
        print(f"{mnq} sin gaps")

    #return problem_indices


### Función para aplicar análisis

In [11]:
def check_df(df, df_name = str):
  nan_count(df, df_name)
  detectar_gaps(df, df_name)

## **2.1. Aplicación de análisis**

In [12]:
check_df (mnq_t2, 'mnq_t2')

mnq_t2 sin valores NaNs
mnq_t2 sin gaps


# **3. Definición de parámetros de división**

Para dividir el dataset en subconjuntos, se utiliza la siguiente estrategia:

- 70% de los días se asignan al conjunto de entrenamiento (train).

- El 30% restante se reparte de manera equitativa entre los conjuntos de validación (valid) y prueba (test).

Esto garantiza que el modelo disponga de la mayor parte de los datos para aprender patrones, mientras que las particiones de validación y prueba permiten ajustar hiperparámetros y evaluar el rendimiento fuera de muestra.

De esta forma, se asegura un esquema de división temporalmente consistente, sin solapamiento entre conjuntos.

#**4. Partición del dataset respetando causalidad temporal**

En este punto, la división del dataset se realiza respetando el orden cronológico de los días, con el objetivo de preservar la causalidad temporal y evitar cualquier forma de data leakage en la evaluación del modelo.

Para ello, se extraen los días únicos presentes en el dataset y se ordenan cronológicamente. A continuación, se asignan los primeros días al conjunto de entrenamiento, los días intermedios al conjunto de validación y los días más recientes al conjunto de prueba, de acuerdo con las proporciones definidas (70 % / 15 % / 15 %).

Este procedimiento garantiza que el modelo sea entrenado exclusivamente con información pasada y evaluado sobre datos futuros, manteniendo la integridad intradía de cada jornada y proporcionando una estimación realista de su capacidad de generalización.

In [13]:
import pandas as pd
from typing import Tuple


def split_intraday_by_day_with_checks(
    df: pd.DataFrame,
    date_col: str = "date",
    minute_col: str = "minute_of_day",
    train_ratio: float = 0.70,
    valid_ratio: float = 0.15,
    test_ratio: float = 0.15,
    verbose: bool = True,
) -> Tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    """
    Verifica integridad temporal del dataset intradía, genera splits por día
    respetando causalidad temporal y valida nuevamente los resultados.

    Reglas:
    - El dataset de entrada debe estar ordenado por (date, minute_of_day)
    - No debe haber duplicados por (date, minute_of_day)
    - El split se hace por días únicos en orden cronológico
    - Los splits resultantes no deben solaparse
    - Cada split debe quedar ordenado internamente

    Retorna
    -------
    mnq_train, mnq_valid, mnq_test
    """

    def _print(msg: str) -> None:
        if verbose:
            print(msg)

    def _validate_sorted_and_unique(data: pd.DataFrame, name: str) -> None:
        _print("=" * 100)
        _print(f"VALIDACIÓN DE ORDEN | {name}")
        _print("=" * 100)

        required_cols = [date_col, minute_col]
        missing = [c for c in required_cols if c not in data.columns]
        if missing:
            raise ValueError(f"{name}: faltan columnas requeridas: {missing}")

        check = data[[date_col, minute_col]].copy()
        check[date_col] = pd.to_datetime(check[date_col])

        sorted_check = check.sort_values([date_col, minute_col], kind="mergesort")

        if check.reset_index(drop=True).equals(sorted_check.reset_index(drop=True)):
            _print(f"[OK] {name}: ordenado por ({date_col}, {minute_col})")
        else:
            raise ValueError(
                f"{name}: el dataset NO está ordenado por ({date_col}, {minute_col})"
            )

        n_dups = check.duplicated(subset=[date_col, minute_col]).sum()
        if n_dups == 0:
            _print(f"[OK] {name}: sin duplicados por ({date_col}, {minute_col})")
        else:
            raise ValueError(
                f"{name}: se encontraron {n_dups} duplicados por ({date_col}, {minute_col})"
            )

        n_days = check[date_col].dt.date.nunique()
        _print(f"[INFO] {name}: rows={len(data):,} | días únicos={n_days:,}")

    def _extract_days(data: pd.DataFrame) -> pd.Series:
        return pd.to_datetime(data[date_col]).dt.date

    # ---------------------------------------------------------
    # 1) Validación dataset de entrada
    # ---------------------------------------------------------
    _validate_sorted_and_unique(df, "DATASET DE ENTRADA")

    total_ratio = train_ratio + valid_ratio + test_ratio
    if abs(total_ratio - 1.0) > 1e-9:
        raise ValueError("train_ratio + valid_ratio + test_ratio debe sumar 1.0")

    all_days = _extract_days(df)
    unique_days = pd.Index(sorted(all_days.unique()))
    n_days = len(unique_days)

    if n_days < 3:
        raise ValueError(
            f"No hay suficientes días para dividir train/valid/test. n_days={n_days}"
        )

    n_train = int(n_days * train_ratio)
    n_valid = int(n_days * valid_ratio)
    n_test = n_days - n_train - n_valid

    if min(n_train, n_valid, n_test) <= 0:
        raise ValueError(
            f"Split inválido: n_train={n_train}, n_valid={n_valid}, n_test={n_test}"
        )

    train_days = set(unique_days[:n_train])
    valid_days = set(unique_days[n_train:n_train + n_valid])
    test_days = set(unique_days[n_train + n_valid:])

    _print("=" * 100)
    _print("GENERACIÓN DE SPLITS")
    _print("=" * 100)
    _print(f"[INFO] Total días  : {n_days}")
    _print(f"[INFO] Train días  : {len(train_days)}")
    _print(f"[INFO] Valid días  : {len(valid_days)}")
    _print(f"[INFO] Test días   : {len(test_days)}")

    mnq_train = df[all_days.isin(train_days)].copy()
    mnq_valid = df[all_days.isin(valid_days)].copy()
    mnq_test = df[all_days.isin(test_days)].copy()

    # ---------------------------------------------------------
    # 2) Validación de cada split
    # ---------------------------------------------------------
    _validate_sorted_and_unique(mnq_train, "TRAIN")
    _validate_sorted_and_unique(mnq_valid, "VALID")
    _validate_sorted_and_unique(mnq_test, "TEST")

    # ---------------------------------------------------------
    # 3) Verificación de no solapamiento
    # ---------------------------------------------------------
    _print("=" * 100)
    _print("VALIDACIÓN DE NO SOLAPAMIENTO ENTRE SPLITS")
    _print("=" * 100)

    train_day_set = set(_extract_days(mnq_train).unique())
    valid_day_set = set(_extract_days(mnq_valid).unique())
    test_day_set = set(_extract_days(mnq_test).unique())

    overlap_train_valid = train_day_set.intersection(valid_day_set)
    overlap_train_test = train_day_set.intersection(test_day_set)
    overlap_valid_test = valid_day_set.intersection(test_day_set)

    if not overlap_train_valid:
        _print("[OK] Train vs Valid: sin solapamiento")
    else:
        raise ValueError(
            f"Hay solapamiento entre train y valid: {sorted(overlap_train_valid)}"
        )

    if not overlap_train_test:
        _print("[OK] Train vs Test : sin solapamiento")
    else:
        raise ValueError(
            f"Hay solapamiento entre train y test: {sorted(overlap_train_test)}"
        )

    if not overlap_valid_test:
        _print("[OK] Valid vs Test : sin solapamiento")
    else:
        raise ValueError(
            f"Hay solapamiento entre valid y test: {sorted(overlap_valid_test)}"
        )

    # ---------------------------------------------------------
    # 4) Verificación de causalidad temporal entre splits
    # ---------------------------------------------------------
    _print("=" * 100)
    _print("VALIDACIÓN DE CAUSALIDAD TEMPORAL ENTRE SPLITS")
    _print("=" * 100)

    train_min, train_max = min(train_day_set), max(train_day_set)
    valid_min, valid_max = min(valid_day_set), max(valid_day_set)
    test_min, test_max = min(test_day_set), max(test_day_set)

    _print(f"[INFO] Train: {train_min} -> {train_max}")
    _print(f"[INFO] Valid: {valid_min} -> {valid_max}")
    _print(f"[INFO] Test : {test_min} -> {test_max}")

    if train_max < valid_min:
        _print("[OK] Todo Train ocurre antes de Valid")
    else:
        raise ValueError("La causalidad temporal entre train y valid no se cumple")

    if valid_max < test_min:
        _print("[OK] Todo Valid ocurre antes de Test")
    else:
        raise ValueError("La causalidad temporal entre valid y test no se cumple")

    if train_max < test_min:
        _print("[OK] Todo Train ocurre antes de Test")
    else:
        raise ValueError("La causalidad temporal entre train y test no se cumple")

    # ---------------------------------------------------------
    # 5) Resumen final
    # ---------------------------------------------------------
    _print("=" * 100)
    _print("RESUMEN FINAL DEL SPLIT")
    _print("=" * 100)
    _print(f"Train rows: {len(mnq_train):,}")
    _print(f"Valid rows: {len(mnq_valid):,}")
    _print(f"Test rows : {len(mnq_test):,}")
    _print("=" * 100)
    _print("Todas las verificaciones pasaron correctamente.")
    _print("=" * 100)

    return mnq_train, mnq_valid, mnq_test

In [14]:
mnq_train, mnq_valid, mnq_test = split_intraday_by_day_with_checks(
    mnq_t2,
    date_col="date",
    minute_col="minute_of_day",
    train_ratio=0.70,
    valid_ratio=0.15,
    test_ratio=0.15,
    verbose=True,
)

VALIDACIÓN DE ORDEN | DATASET DE ENTRADA
[OK] DATASET DE ENTRADA: ordenado por (date, minute_of_day)
[OK] DATASET DE ENTRADA: sin duplicados por (date, minute_of_day)
[INFO] DATASET DE ENTRADA: rows=88,920 | días únicos=1,482
GENERACIÓN DE SPLITS
[INFO] Total días  : 1482
[INFO] Train días  : 1037
[INFO] Valid días  : 222
[INFO] Test días   : 223
VALIDACIÓN DE ORDEN | TRAIN
[OK] TRAIN: ordenado por (date, minute_of_day)
[OK] TRAIN: sin duplicados por (date, minute_of_day)
[INFO] TRAIN: rows=62,220 | días únicos=1,037
VALIDACIÓN DE ORDEN | VALID
[OK] VALID: ordenado por (date, minute_of_day)
[OK] VALID: sin duplicados por (date, minute_of_day)
[INFO] VALID: rows=13,320 | días únicos=222
VALIDACIÓN DE ORDEN | TEST
[OK] TEST: ordenado por (date, minute_of_day)
[OK] TEST: sin duplicados por (date, minute_of_day)
[INFO] TEST: rows=13,380 | días únicos=223
VALIDACIÓN DE NO SOLAPAMIENTO ENTRE SPLITS
[OK] Train vs Valid: sin solapamiento
[OK] Train vs Test : sin solapamiento
[OK] Valid vs Test

#**5. Generación de Summaryl**

In [15]:
import json
from pathlib import Path
import pandas as pd


def generate_splits_summary(
    mnq_train: pd.DataFrame,
    mnq_valid: pd.DataFrame,
    mnq_test: pd.DataFrame,
    features_col,
    targets_col,
    aux_col,
    out_path: Path,
    date_col: str = "date",
    regime_col: str = "regime_id",
    verbose: bool = True,
):
    """
    Genera un summary JSON de los splits (train/valid/test)
    """

    def _print(msg):
        if verbose:
            print(msg)

    def _get_split_summary(df: pd.DataFrame, name: str):

        dates = pd.to_datetime(df[date_col])

        summary = {
            "shape": {
                "n_rows": int(len(df)),
                "n_cols": int(df.shape[1]),
            },
            "time_range": {
                "start": str(dates.min()),
                "end": str(dates.max()),
            },
            "n_sessions": int(dates.dt.date.nunique()),
        }

        # -------------------------
        # NaNs
        # -------------------------
        nan_report = {}
        for group_name, cols in {
            "auxiliary": aux_col,
            "features": features_col,
            "targets": targets_col,
        }.items():

            cols_present = [c for c in cols if c in df.columns]

            nan_report[group_name] = {
                "n_nan_total": int(df[cols_present].isna().sum().sum()),
                "n_nan_by_col": {
                    c: int(df[c].isna().sum()) for c in cols_present
                },
            }

        summary["nan_report"] = nan_report

        # -------------------------
        # Targets distribución
        # -------------------------
        target_dist = {}

        for t in targets_col:
            if t not in df.columns:
                continue

            counts = df[t].value_counts(normalize=False)
            total = counts.sum()

            target_dist[t] = {
                str(k): {
                    "count": int(v),
                    "pct": float(v / total),
                }
                for k, v in counts.items()
            }

        summary["target_distribution"] = target_dist

        # -------------------------
        # Régimen distribución
        # -------------------------
        if regime_col in df.columns:
            reg_counts = df[regime_col].value_counts()
            total = reg_counts.sum()

            summary["regime_distribution"] = {
                str(k): {
                    "count": int(v),
                    "pct": float(v / total),
                }
                for k, v in reg_counts.items()
            }

        return summary

    # ---------------------------------------------------
    # Construcción del summary completo
    # ---------------------------------------------------
    splits_summary = {
        "train": _get_split_summary(mnq_train, "train"),
        "valid": _get_split_summary(mnq_valid, "valid"),
        "test": _get_split_summary(mnq_test, "test"),
    }

    # Guardar
    out_path.parent.mkdir(parents=True, exist_ok=True)

    with open(out_path, "w") as f:
        json.dump(splits_summary, f, indent=4)

    _print("=" * 100)
    _print("SPLITS SUMMARY GENERADO")
    _print(f"Path: {out_path}")
    _print("=" * 100)

    return splits_summary

In [16]:
OUT_SPLITS_SUMMARY

PosixPath('/content/drive/MyDrive/neural_profit/data/05_splits/splits_summary.json')

In [17]:
splits_summary = generate_splits_summary(
    mnq_train,
    mnq_valid,
    mnq_test,
    features_col=features_col,
    targets_col=targets_col,
    aux_col=aux_col,
    out_path=OUT_SPLITS_SUMMARY,
)

SPLITS SUMMARY GENERADO
Path: /content/drive/MyDrive/neural_profit/data/05_splits/splits_summary.json


# **6. Guardamos los datasets generados**


In [18]:
from pathlib import Path
import pandas as pd


def prepare_and_save(
    df: pd.DataFrame,
    path: Path,
    date_col: str = "date",
    minute_col: str = "minute_of_day",
):
    """
    Ordena, valida y guarda preservando el DatetimeIndex.
    """

    if not isinstance(df.index, pd.DatetimeIndex):
        raise TypeError(
            f"Se requiere DatetimeIndex antes de guardar. Recibido: {type(df.index)}"
        )

    # Orden explícito por sesión y minuto
    df_out = df.sort_values([date_col, minute_col]).copy()

    # Validar orden lógico
    check = df_out[[date_col, minute_col]]
    if not check.reset_index(drop=True).equals(
        check.sort_values([date_col, minute_col]).reset_index(drop=True)
    ):
        raise ValueError("El dataset no está ordenado por (date, minute_of_day)")

    # Validar orden del índice también
    if not df_out.index.is_monotonic_increasing:
        raise ValueError("El DatetimeIndex no está ordenado de forma ascendente")

    # Guardar preservando índice
    path.parent.mkdir(parents=True, exist_ok=True)
    df_out.to_parquet(path, index=True)

    print(f"[OK] Guardado con DatetimeIndex preservado: {path}")
    return df_out


In [19]:
mnq_train = prepare_and_save(mnq_train, OUT_PARQUET_T2_TRAIN)
mnq_valid = prepare_and_save(mnq_valid, OUT_PARQUET_T2_VALID)
mnq_test  = prepare_and_save(mnq_test,  OUT_PARQUET_T2_TEST)

[OK] Guardado con DatetimeIndex preservado: /content/drive/MyDrive/neural_profit/data/05_splits/mnq_t2_train.parquet
[OK] Guardado con DatetimeIndex preservado: /content/drive/MyDrive/neural_profit/data/05_splits/mnq_t2_valid.parquet
[OK] Guardado con DatetimeIndex preservado: /content/drive/MyDrive/neural_profit/data/05_splits/mnq_t2_test.parquet


# **7. Alineación con libro ML**


La separación de los datos se realiza respetando estrictamente el orden temporal,
de acuerdo con las buenas prácticas de *Machine Learning* para series temporales
financieras intradía.

### Principio fundamental

En problemas de series temporales **no se permiten splits aleatorios**.
El tiempo siempre fluye en una única dirección, por lo que:

- El conjunto de entrenamiento contiene únicamente información pasada.
- El conjunto de validación representa un período posterior e independiente.
- El conjunto de test simula el comportamiento futuro del modelo.

---

### Esquema de separación

- El split se realiza **por jornadas completas**, no por minutos individuales.
- Cada subconjunto contiene **días consecutivos**, sin solapamientos.
- No se mezclan observaciones de una misma jornada entre distintos splits.

De este modo, se evita cualquier fuga de información entre conjuntos.

---

### Rol de cada subconjunto

- **Train**  
  Utilizado para el entrenamiento del modelo y el ajuste de parámetros internos.

- **Validation**  
  Utilizado para:
  - selección de hiperparámetros,
  - comparación entre modelos,
  - decisiones de arquitectura.

- **Test**  
  Reservado exclusivamente para la evaluación final.
  No participa en ninguna decisión previa y se evalúa **una sola vez**.

---

### Consistencia temporal y de horizontes

- El criterio de separación es **idéntico** para los horizontes  
  **H = 60 minutos** y **H = 90 minutos**.
- La única diferencia entre ambos problemas es el horizonte del target,
  manteniéndose constantes:
  - las fechas de corte,
  - las jornadas incluidas,
  - la lógica de descarte de observaciones finales por día.

---

### Consideraciones intradía

- Los últimos minutos de cada jornada se descartan al construir los targets
  cuando no existe información suficiente hacia adelante para el horizonte definido.
- Esta regla se aplica de manera consistente en *train*, *validation* y *test*.

---

### Diagnóstico

La separación de datos implementa un **time-aware split correcto**, coherente con
el carácter secuencial del problema y adecuada para evaluar la capacidad de
generalización del modelo en un contexto intradía realista.

Este bloque **cierra el Punto 3** del proceso de *Machine Learning*.
